# 🚀 Fake News Detection System — Full Pipeline + Docker Deployment
### Trains 4 models (LR, DT, GB, RF) and packages everything for Docker


## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn joblib fastapi uvicorn pydantic

## 📂 Step 2 — Load Dataset

In [ ]:
import pandas as pd
import numpy as np

# Load your news.csv file
# Expected columns: 'text' and 'label' (0=FAKE, 1=REAL)
df = pd.read_csv('news.csv')

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nSample:')
display(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts())

## 🧹 Step 3 — Text Cleaning

In [ ]:
import re

def wordopt(text):
    """Clean and normalize text for fake news detection."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)          # remove [brackets]
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'<.*?>+', '', text)            # remove HTML tags
    text = re.sub(r'[^\w\s]', '', text)           # remove punctuation
    text = re.sub(r'\n', ' ', text)               # remove newlines
    text = re.sub(r'\w*\d\w*', '', text)          # remove words with numbers
    text = re.sub(r'\s+', ' ', text).strip()      # remove extra spaces
    return text

# Apply cleaning
df['text_clean'] = df['text'].apply(wordopt)

# Drop rows with empty text after cleaning
df = df[df['text_clean'].str.strip() != '']
df = df.dropna(subset=['text_clean', 'label'])

print(f'✅ Cleaned dataset: {df.shape}')
print('\nSample cleaned text:')
print(df['text_clean'].iloc[0][:200])

## ✂️ Step 4 — Split Data

In [ ]:
from sklearn.model_selection import train_test_split

X = df['text_clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')
print(f'\nTrain label distribution:\n{y_train.value_counts()}')

## 🔢 Step 5 — TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    stop_words='english'
)

X_train_vec = vectorization.fit_transform(X_train)
X_test_vec  = vectorization.transform(X_test)

print(f'✅ Vectorizer fitted')
print(f'   Vocabulary size: {len(vectorization.vocabulary_)}')
print(f'   Feature matrix:  {X_train_vec.shape}')

## 🤖 Step 6 — Train All 4 Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

models = {
    'LR': LogisticRegression(max_iter=1000, random_state=42),
    'DT': DecisionTreeClassifier(random_state=42),
    'GB': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'RF': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

trained = {}
results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    trained[name] = model
    results[name] = acc
    print(f'✅  Accuracy: {acc:.4f}')

print('\n📊 Summary:')
for name, acc in results.items():
    print(f'   {name}: {acc*100:.2f}%')

## 📊 Step 7 — Detailed Evaluation

In [ ]:
best_model_name = max(results, key=results.get)
best_model = trained[best_model_name]

print(f'🏆 Best Model: {best_model_name} ({results[best_model_name]*100:.2f}%)')
print('\n📋 Classification Report (Best Model):')
y_pred_best = best_model.predict(X_test_vec)
print(classification_report(y_test, y_pred_best, target_names=['FAKE', 'REAL']))

## 💾 Step 8 — Save All Models & Vectorizer

In [ ]:
import os
import joblib

SAVE_DIR = 'FAKE_NEWS_DETECTION'
os.makedirs(SAVE_DIR, exist_ok=True)

# Save all 4 models
joblib.dump(trained['LR'], f'{SAVE_DIR}/lr_model.joblib')
joblib.dump(trained['DT'], f'{SAVE_DIR}/dt_model.joblib')
joblib.dump(trained['GB'], f'{SAVE_DIR}/gb_model.joblib')
joblib.dump(trained['RF'], f'{SAVE_DIR}/rf_model.joblib')

# Save vectorizer
joblib.dump(vectorization, f'{SAVE_DIR}/vectorizer.joblib')

print(f'✅ All models saved to {SAVE_DIR}/')
print('\nFiles saved:')
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{f}')
    print(f'  📁 {f}  →  {size/1024:.1f} KB')

## 📝 Step 9 — Create app.py

In [ ]:
app_code = '''
import os
import re
from collections import Counter

import joblib
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Fake News Detection API",
    description="Detects fake news using ensemble of 4 ML models (LR, DT, GB, RF)",
    version="1.0"
)

FAKE_NEWS_DETECTION = "FAKE_NEWS_DETECTION"

# Load all models
lr            = joblib.load(f"{FAKE_NEWS_DETECTION}/lr_model.joblib")
dt            = joblib.load(f"{FAKE_NEWS_DETECTION}/dt_model.joblib")
gb            = joblib.load(f"{FAKE_NEWS_DETECTION}/gb_model.joblib")
rf            = joblib.load(f"{FAKE_NEWS_DETECTION}/rf_model.joblib")
vectorization = joblib.load(f"{FAKE_NEWS_DETECTION}/vectorizer.joblib")

print("✅ All models loaded successfully")


def wordopt(text: str) -> str:
    """Clean and normalize text."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\\[.*?\\]", "", text)
    text = re.sub(r"https?://\\S+|www\\.\\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[^\\w\\s]", "", text)
    text = re.sub(r"\\n", " ", text)
    text = re.sub(r"\\w*\\d\\w*", "", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text


label_map = {0: "FAKE", 1: "REAL"}


class NewsRequest(BaseModel):
    text: str


class PredictionResponse(BaseModel):
    input_text_preview: str
    model_votes: dict
    overall_verdict: str
    agreement: str
    confidence: str


@app.get("/")
def home():
    return {
        "message": "Fake News Detection API is running! ✅",
        "usage": "POST /predict with JSON body: {\\"text\\": \\"your news article\\"}",
        "models": ["Logistic Regression", "Decision Tree", "Gradient Boosting", "Random Forest"]
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(payload: NewsRequest):
    # Clean text
    cleaned = wordopt(payload.text)

    if not cleaned:
        return {
            "input_text_preview": "",
            "model_votes": {},
            "overall_verdict": "ERROR",
            "agreement": "0/4",
            "confidence": "0%"
        }

    # Vectorize
    xv = vectorization.transform([cleaned])

    # Get predictions from all 4 models
    preds = {
        "LR": label_map[int(lr.predict(xv)[0])],
        "DT": label_map[int(dt.predict(xv)[0])],
        "GB": label_map[int(gb.predict(xv)[0])],
        "RF": label_map[int(rf.predict(xv)[0])],
    }

    # Majority vote
    votes = list(preds.values())
    overall_verdict, count = Counter(votes).most_common(1)[0]

    return {
        "input_text_preview": payload.text[:100] + "...",
        "model_votes": preds,
        "overall_verdict": overall_verdict,
        "agreement": f"{count}/4 models agree",
        "confidence": f"{(count / 4) * 100:.0f}%"
    }


@app.get("/health")
def health():
    return {"status": "healthy", "models_loaded": 4}
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('✅ app.py created!')
!cat app.py

## 🐳 Step 10 — Create Dockerfile

In [ ]:
dockerfile = '''# ---- Base image ----
FROM python:3.11-slim

WORKDIR /app

# System dependencies
RUN apt-get update && apt-get install -y --no-install-recommends \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Install Python dependencies first (better layer caching)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy app and models
COPY app.py .
COPY FAKE_NEWS_DETECTION/ ./FAKE_NEWS_DETECTION/

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

print('✅ Dockerfile created!')
!cat Dockerfile

## 📋 Step 11 — Create requirements.txt

In [ ]:
requirements = '''fastapi==0.111.0
uvicorn==0.30.1
pydantic==2.7.1
scikit-learn==1.5.0
joblib==1.4.2
numpy==1.26.4
pandas==2.2.2
'''

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print('✅ requirements.txt created!')
!cat requirements.txt

## 🙈 Step 12 — Create .dockerignore

In [ ]:
dockerignore = '''__pycache__/
*.pyc
*.pyo
*.ipynb
*.zip
*.csv
DATASET/
DATASET.zip
.ipynb_checkpoints/
'''

with open('.dockerignore', 'w') as f:
    f.write(dockerignore)

print('✅ .dockerignore created!')

## 🧪 Step 13 — Test the App Locally in Colab

In [ ]:
# Quick local test before Docker
import re
import joblib
from collections import Counter

FAKE_NEWS_DETECTION = 'FAKE_NEWS_DETECTION'

lr_test = joblib.load(f'{FAKE_NEWS_DETECTION}/lr_model.joblib')
dt_test = joblib.load(f'{FAKE_NEWS_DETECTION}/dt_model.joblib')
gb_test = joblib.load(f'{FAKE_NEWS_DETECTION}/gb_model.joblib')
rf_test = joblib.load(f'{FAKE_NEWS_DETECTION}/rf_model.joblib')
vec_test = joblib.load(f'{FAKE_NEWS_DETECTION}/vectorizer.joblib')

def wordopt_test(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

label_map = {0: 'FAKE', 1: 'REAL'}

def test_prediction(news_text):
    cleaned = wordopt_test(news_text)
    xv = vec_test.transform([cleaned])
    preds = {
        'LR': label_map[int(lr_test.predict(xv)[0])],
        'DT': label_map[int(dt_test.predict(xv)[0])],
        'GB': label_map[int(gb_test.predict(xv)[0])],
        'RF': label_map[int(rf_test.predict(xv)[0])],
    }
    votes = list(preds.values())
    verdict, count = Counter(votes).most_common(1)[0]
    print(f'\n📰 Text: {news_text[:80]}...')
    print(f'🔍 Model votes: {preds}')
    print(f'🏁 Verdict: {verdict} ({count}/4 models agree) — {(count/4)*100:.0f}% confidence')

# Test with sample news
test_prediction("Scientists have discovered a new vaccine that can prevent all types of cancer with 99% effectiveness in clinical trials")
test_prediction("Aliens have landed in the White House and taken over the government according to secret sources")

## 📦 Step 14 — Verify All Files Exist

In [ ]:
import os

required_files = [
    'app.py',
    'Dockerfile',
    'requirements.txt',
    '.dockerignore',
    'FAKE_NEWS_DETECTION/lr_model.joblib',
    'FAKE_NEWS_DETECTION/dt_model.joblib',
    'FAKE_NEWS_DETECTION/gb_model.joblib',
    'FAKE_NEWS_DETECTION/rf_model.joblib',
    'FAKE_NEWS_DETECTION/vectorizer.joblib',
]

print('📋 File Check:')
all_good = True
for f in required_files:
    exists = os.path.exists(f)
    size = os.path.getsize(f) / 1024 if exists else 0
    status = '✅' if exists else '❌'
    print(f'  {status} {f}  ({size:.1f} KB)')
    if not exists:
        all_good = False

if all_good:
    print('\n🎉 All files ready for Docker!')
else:
    print('\n⚠️ Some files are missing. Run the steps above again.')

## ⬇️ Step 15 — Zip & Download Everything to Your PC

In [ ]:
# Zip all required files
!zip -r fake-news-docker.zip \
    app.py \
    Dockerfile \
    requirements.txt \
    .dockerignore \
    FAKE_NEWS_DETECTION/

# Show zip size
!du -sh fake-news-docker.zip

# Download to your PC
from google.colab import files
files.download('fake-news-docker.zip')

print('✅ Downloaded! Now extract and run Docker commands on your PC.')

## 🐳 Step 16 — Docker Commands (Run on Your PC Terminal)

After downloading and extracting `fake-news-docker.zip`, open terminal in the extracted folder and run:

```bash
# 1. Go to extracted folder
cd /path/to/fake-news-docker

# 2. Login to Docker Hub
docker login -u pramana31
# Enter your access token: dckr_pat_...

# 3. Build Docker image
docker build -t fake-news-detection-system .

# 4. Test locally
docker run -p 8000:8000 fake-news-detection-system
# Open: http://localhost:8000

# 5. Tag for Docker Hub
docker tag fake-news-detection-system pramana31/fake-news-detection-system:latest

# 6. Push to Docker Hub
docker push pramana31/fake-news-detection-system:latest
```

### Test API after running:
```bash
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{"text": "Scientists discover cure for cancer in major breakthrough"}'
```

### Interactive API Docs:
Open browser → `http://localhost:8000/docs`
